# high-level pseudocode (Python-like)

for config in grid:
    # 1) generate datasets
    train, val, test = simulate_lob_dataset(config, days=30, window=100)

    # 2) build inputs (offset-form)
    X_train, y_train = build_offset_windows(train, H=H)  # sliding windows
    X_val, y_val   = build_offset_windows(val, H=H)
    X_test, y_test = build_offset_windows(test, H=H)
    save_Xy(...)  # optional

    # 3) model
    model = build_deeplob_like_model(input_shape=(100,40))
    model.fit(X_train, y_train, validation=(X_val,y_val), epochs=20, batch=128)

    # 4) test predictions
    probs = model.predict_proba(X_test)
    preds = probs.argmax(axis=1)
    confidences = probs.max(axis=1)

    # 5) evaluate over thresholds
    results = []
    for t in thresholds:
        idx = confidences >= t
        coverage = idx.mean()
        if coverage < 0.01:
            mcc = None
        else:
            mcc = matthews_corrcoef(y_test[idx], preds[idx])
        results.append((t, coverage, mcc))
    save_results(config, results, probs, preds, confidences)

# 6) aggregated plotting: MCC vs threshold, coverage vs threshold,
#    separate rows by tick-size (small/medium/large) and columns by horizon (H10/H50/H100)
